# 研究与工程思维 3/6：对照实验、消融、混杂与交互

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | 改了很多东西并且分数上涨，怎样知道究竟什么有效？ |
| 迁移价值 | 适用于调参、重构、A/B 测试、音频处理链和性能优化。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：改了很多东西并且分数上涨，怎样知道究竟什么有效？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [NIST：为什么单因素逐次优化会漏掉交互](https://www.itl.nist.gov/div898/handbook/pri/section2/pri212.htm)
- [NIST：如何选择实验设计](https://www.itl.nist.gov/div898/handbook/pri/section3/pri3.htm)
- [NIST：随机化与重复的作用](https://www.itl.nist.gov/div898/handbook/pmd/section3/pmd33.htm)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. 公平对照的最低合同

只改变计划研究的因素，并固定：数据与划分、文本规范化、随机种子策略、训练预算、解码参数、硬件/线程、评分代码。若无法固定，就记录为阻断因素或协变量。

“新版模型 + 新前端 + 新 LM 在新测试集上更好”没有可归因性。它可以是有用的工程候选，但不是组件有效性的证据。


In [ ]:
from itertools import product

# 两因素全因子：前端 F、语言模型 L。数值为教学 WER（越低越好）。
wer = {
    (0, 0): 20.0,
    (1, 0): 17.0,
    (0, 1): 18.0,
    (1, 1): 11.0,
}

for f, l in product([0, 1], repeat=2):
    print(f"frontend={f}, lm={l}, WER={wer[(f,l)]:.1f}%")

frontend_effect_without_lm = wer[(1, 0)] - wer[(0, 0)]
frontend_effect_with_lm = wer[(1, 1)] - wer[(0, 1)]
interaction = frontend_effect_with_lm - frontend_effect_without_lm
print("前端在无LM时的变化:", frontend_effect_without_lm, "pp")
print("前端在有LM时的变化:", frontend_effect_with_lm, "pp")
print("交互项:", interaction, "pp")
assert interaction == -4.0


## 2. 为什么“每次只改一个变量”不是完整规则

排错时，一次改变一个变量有利于归因；但探索多个因素时，纯 OFAT 会漏掉交互。上例的前端与 LM 一起使用有额外收益。正确升级是：

- 小范围定位：最小配对对照；
- 多因素筛选：全因子或设计良好的部分因子；
- 存在批次/说话人差异：分块；
- 存在时间漂移：随机化运行顺序并重复。


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
conditions = np.array([0] * 10 + [1] * 10)  # 0=baseline, 1=candidate
time_drift = np.linspace(0, 4, len(conditions))
true_effect = -1.5

def observed_difference(order):
    y = 20 + true_effect * conditions[order] + time_drift + rng.normal(0, .15, len(order))
    assigned = conditions[order]
    return y[assigned == 1].mean() - y[assigned == 0].mean()

blocked_order = np.arange(20)  # baseline 全在前，candidate 全在后：与漂移混杂
random_order = np.random.default_rng(3).permutation(20)
print("未随机化估计:", round(observed_difference(blocked_order), 2), "pp")
print("随机化估计:", round(observed_difference(random_order), 2), "pp")
print("真实干预效应:", true_effect, "pp")


## 3. 消融表必须回答机制问题

| 实验 | 前端 | LM | 训练预算 | 测试集 | 目的 |
|---|---:|---:|---:|---|---|
| E0 | 0 | 0 | 固定 | 固定 | 最弱合理基线 |
| E1 | 1 | 0 | 固定 | 固定 | 前端主效应 |
| E2 | 0 | 1 | 固定 | 固定 | LM 主效应 |
| E3 | 1 | 1 | 固定 | 固定 | 完整系统与交互 |

若完整系统只赢 0.2pp，却多 40% 延迟，结论不能只写“最优”。还需要统计区间和工程约束。


In [ ]:
def audit_experiment(base: dict, candidate: dict, intended_factor: str):
    keys = sorted(set(base) | set(candidate))
    changed = [key for key in keys if base.get(key) != candidate.get(key)]
    confounds = [key for key in changed if key != intended_factor]
    return changed, confounds

base = {"frontend": "v1", "test": "fixed", "beam": 10, "threads": 1, "seed": 7}
candidate = {"frontend": "v2", "test": "new", "beam": 20, "threads": 1, "seed": 7}
changed, confounds = audit_experiment(base, candidate, "frontend")
print("发生变化:", changed)
print("混杂因素:", confounds)
assert confounds == ["beam", "test"]


## 闭卷挑战

为“降噪、VAD、LM”设计一个 2×2×2 实验。写出响应变量、硬约束、随机化/分块方法、重复次数和交互项；再说明如果只能跑 4 次，你愿意牺牲什么结论。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：第 4 课学习点估计、Bootstrap 配对区间和置信度校准。
